# MovieLens 10M — Data Cleaning

Load `movies.dat`, `ratings.dat`, and `tags.dat`, check for data-quality issues, drop bad rows, and save cleaned CSVs.
A log of everything dropped is kept in `cleaning_log.txt`.

The 10M dataset has no `users.dat` (anonymized IDs only), uses UTF-8 encoding, and allows half-star ratings.

In [1]:
import pandas as pd
from pathlib import Path

DATA_DIR = Path('../dataset')
OUT_DIR = Path('cleaned')
OUT_DIR.mkdir(exist_ok=True)

log_lines = []
def log(msg):
    print(msg)
    log_lines.append(msg)

## 1. Load raw files

Files use `::` as separator, have no header, and are UTF-8 encoded.

In [2]:
movies = pd.read_csv(
    DATA_DIR / 'movies-1M.dat',
    sep='::', engine='python', header=None,
    names=['MovieID', 'Title', 'Genres'],
    encoding='utf-8',
    low_memory=False
)

ratings = pd.read_csv(
    DATA_DIR / 'ratings.dat',
    sep='::', engine='python', header=None,
    names=['UserID', 'MovieID', 'Rating', 'Timestamp'],
    encoding='utf-8',
    low_memory=False
)

tags = pd.read_csv(
    DATA_DIR / 'tags.dat',
    sep='::', engine='python', header=None,
    names=['UserID', 'MovieID', 'Tag', 'Timestamp'],
    encoding='utf-8',
    low_memory=False
)

log(f'Loaded: movies-1M={len(movies):,}, ratings={len(ratings):,}, tags={len(tags):,}')

Loaded: movies=10,681, ratings=10,000,054, tags=95,580


In [3]:
movies.head()

,MovieID,Title,Genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [4]:
ratings.head()

,UserID,MovieID,Rating,Timestamp
0,1,122,5.0,838985046
1,1,185,5.0,838983525
2,1,231,5.0,838983392
3,1,292,5.0,838983421
4,1,316,5.0,838983392


In [5]:
tags.head()

,UserID,MovieID,Tag,Timestamp
0,15,4973,excellent!,1215184630
1,20,1747,politics,1188263867
2,20,1747,satire,1188263867
3,20,2424,chick flick 212,1188263835
4,20,2424,hanks,1188263835


## 2. Missing values

Any row with a null in any field gets dropped. Blank/whitespace-only strings are treated as nulls.

In [6]:
for name, df in [('movies-1M', movies), ('ratings', ratings), ('tags', tags)]:
    log(f'Nulls in {name}:\n{df.isna().sum().to_string()}\n')

Nulls in movies:
MovieID    0
Title      0
Genres     0

Nulls in ratings:
UserID       0
MovieID      0
Rating       0
Timestamp    0

Nulls in tags:
UserID        0
MovieID       0
Tag          16
Timestamp     0



In [7]:
# Normalize strings first: strip whitespace and treat empty strings as null,
# so blank/whitespace-only fields are dropped along with real nulls.
for df, cols in [(movies, ['Title', 'Genres']),
                 (ratings, []),
                 (tags, ['Tag'])]:
    for c in cols:
        df[c] = df[c].astype(str).str.strip().replace({'': None, 'nan': None})

before = len(movies), len(ratings), len(tags)
movies = movies.dropna()
ratings = ratings.dropna()
tags = tags.dropna()
log(f'Dropped nulls/blanks: movies-1M={before[0]-len(movies)}, '
    f'ratings={before[1]-len(ratings)}, tags={before[2]-len(tags)}')

Dropped nulls/blanks: movies=0, ratings=0, tags=22


## 3. Duplicates

- Exact duplicate rows
- Duplicate primary keys (`MovieID`, `(UserID, MovieID)` for ratings, `(UserID, MovieID, Tag)` for tags)

In [8]:
before = len(movies)
movies = movies.drop_duplicates().drop_duplicates(subset='MovieID')
log(f'movies-1M: dropped {before - len(movies)} duplicate rows')

# Ratings: first drop fully-identical rows, then detect conflicting (UserID, MovieID)
# pairs — i.e. the same user rated the same movie twice with different Rating/Timestamp.
# For those, keep one row arbitrarily so each (UserID, MovieID) tuple has exactly one rating.
before = len(ratings)
ratings = ratings.drop_duplicates()
log(f'ratings: dropped {before - len(ratings)} exact-duplicate rows')

before = len(ratings)
conflict_mask = ratings.duplicated(subset=['UserID', 'MovieID'], keep=False)
n_conflicts = conflict_mask.sum()
ratings = ratings.drop_duplicates(subset=['UserID', 'MovieID'])
log(f'ratings: found {n_conflicts} rows in conflicting (UserID, MovieID) groups; '
    f'dropped {before - len(ratings)} to keep one rating per tuple')

# Tags: a (UserID, MovieID) pair can legitimately have multiple different tags,
# so the primary key is (UserID, MovieID, Tag). Drop exact duplicates on that tuple.
before = len(tags)
tags = tags.drop_duplicates().drop_duplicates(subset=['UserID', 'MovieID', 'Tag'])
log(f'tags: dropped {before - len(tags)} duplicate rows')

movies: dropped 0 duplicate rows


ratings: dropped 0 exact-duplicate rows


ratings: found 0 rows in conflicting (UserID, MovieID) groups; dropped 0 to keep one rating per tuple
tags: dropped 0 duplicate rows


## 4. Type & value validation

Coerce numeric columns, then drop rows where the value is now `NaN` (meaning the original was malformed) or outside the expected range.

In [9]:
# Movies:
#   - MovieID: positive integer in 1..65133 (observed range in 10M)
#   - Title: must end with a 4-digit year in parentheses, e.g. "Toy Story (1995)"
#   - Genres: pipe-separated tokens from the 10M whitelist (README's 18 genres
#     minus "Children's" which the data spells "Children", plus "IMAX" which
#     appears in the data but not the README)
#   - The literal token "(no genres listed)" appears for movies-1M with unknown genres
#     — we drop those rows to stay consistent with the 1M notebook's policy of
#     requiring valid genre tokens on every movie
#   - Titles/Genres must contain only ASCII letters, digits, and punctuation
#     (accented foreign-film titles are transliterated, not dropped)
import unicodedata

VALID_GENRES = {
    "Action", "Adventure", "Animation", "Children", "Comedy", "Crime",
    "Documentary", "Drama", "Fantasy", "Film-Noir", "Horror", "IMAX",
    "Musical", "Mystery", "Romance", "Sci-Fi", "Thriller", "War", "Western",
}

# Transliterate non-ASCII (é→e, å→a, ³→3, æ→ae, ø→o) so every char is English/numeric
LIGATURES = {'æ': 'ae', 'Æ': 'AE', 'ø': 'o', 'Ø': 'O', 'ß': 'ss'}
def to_ascii(s):
    s = ''.join(LIGATURES.get(c, c) for c in str(s))
    return unicodedata.normalize('NFKD', s).encode('ascii', 'ignore').decode('ascii')

title_changed = (~movies['Title'].map(str.isascii)).sum()
genre_changed = (~movies['Genres'].map(str.isascii)).sum()
movies['Title'] = movies['Title'].apply(to_ascii)
movies['Genres'] = movies['Genres'].apply(to_ascii)
log(f'movies-1M: transliterated {title_changed} titles and {genre_changed} genres to ASCII')

before = len(movies)
movies['MovieID'] = pd.to_numeric(movies['MovieID'], errors='coerce')
movies = movies.dropna(subset=['MovieID'])
movies['MovieID'] = movies['MovieID'].astype(int)
movies = movies[(movies['MovieID'] >= 1) & (movies['MovieID'] <= 65133)]
log(f'movies-1M: dropped {before - len(movies)} rows with invalid MovieID')

# Title must end with "(YYYY)" — extract the year into its own column, strip from Title
before = len(movies)
year_extract = movies['Title'].str.extract(r'^(.*)\s*\((\d{4})\)\s*$')
year_extract.columns = ['TitleOnly', 'Year']
valid = year_extract['Year'].notna()
movies = movies.loc[valid].copy()
movies['Title'] = year_extract.loc[valid, 'TitleOnly'].str.strip().values
movies['Year'] = year_extract.loc[valid, 'Year'].astype(int).values
log(f'movies-1M: dropped {before - len(movies)} rows with malformed Title (missing year); '
    f'extracted Year column (range {movies["Year"].min()}..{movies["Year"].max()})')

# Drop movies-1M flagged "(no genres listed)" — no usable genre information
before = len(movies)
movies = movies[movies['Genres'] != '(no genres listed)']
log(f'movies-1M: dropped {before - len(movies)} rows with "(no genres listed)"')

# Every genre token must be in the whitelist
before = len(movies)
def genres_ok(g):
    parts = str(g).split('|')
    return len(parts) > 0 and all(p in VALID_GENRES for p in parts)
movies = movies[movies['Genres'].apply(genres_ok)]
log(f'movies-1M: dropped {before - len(movies)} rows with unknown genre tokens')

# Enforce ASCII-only: drop any row where Title or Genres still has a non-ASCII char
before = len(movies)
movies = movies[movies['Title'].map(str.isascii) & movies['Genres'].map(str.isascii)]
log(f'movies-1M: dropped {before - len(movies)} rows with non-ASCII characters (post-transliteration)')

# Split pipe-separated Genres into Genre1..GenreN columns (empty string for unused slots)
max_genres = movies['Genres'].str.split('|').map(len).max()
genre_cols = [f'Genre{i+1}' for i in range(max_genres)]
split_df = movies['Genres'].str.split('|', expand=True).fillna('')
split_df.columns = genre_cols
movies = pd.concat([movies.drop(columns='Genres'), split_df], axis=1)
# Put Year right after Title for readability
movies = movies[['MovieID', 'Title', 'Year'] + genre_cols]
log(f'movies-1M: split Genres into {max_genres} columns: {genre_cols}')

movies: transliterated 386 titles and 0 genres to ASCII
movies: dropped 0 rows with invalid MovieID
movies: dropped 0 rows with malformed Title (missing year); extracted Year column (range 1915..2008)
movies: dropped 1 rows with "(no genres listed)"
movies: dropped 0 rows with unknown genre tokens
movies: dropped 0 rows with non-ASCII characters (post-transliteration)
movies: split Genres into 8 columns: ['Genre1', 'Genre2', 'Genre3', 'Genre4', 'Genre5', 'Genre6', 'Genre7', 'Genre8']


In [10]:
# Ratings:
#   - Rating is on a half-star scale: multiples of 0.5 in 0.5..5.0
#   - UserID in 1..71567, MovieID in 1..65133 (observed ranges in 10M)
#   - Timestamp: seconds since epoch, sanity-bounded to [MovieLens launch, today]
before = len(ratings)
for col in ['UserID', 'MovieID', 'Rating', 'Timestamp']:
    ratings[col] = pd.to_numeric(ratings[col], errors='coerce')
ratings = ratings.dropna()

# Half-star check: reject anything that isn't a multiple of 0.5
ratings = ratings[(ratings['Rating'] * 2) % 1 == 0]
ratings = ratings.astype({'UserID': int, 'MovieID': int, 'Timestamp': int})
ratings['Rating'] = ratings['Rating'].astype(float)

ratings = ratings[(ratings['Rating'] >= 0.5) & (ratings['Rating'] <= 5.0)]
ratings = ratings[(ratings['UserID'] >= 1) & (ratings['UserID'] <= 71567)]
ratings = ratings[(ratings['MovieID'] >= 1) & (ratings['MovieID'] <= 65133)]

# Timestamp sanity: between 1990-01-01 and now. The 10M dataset contains
# ratings going back to Jan 1995 (seeded from systems that predate MovieLens
# itself), so the 1M notebook's 1997 floor is too aggressive here.
import time
TS_MIN, TS_MAX = 631152000, int(time.time())
ratings = ratings[(ratings['Timestamp'] >= TS_MIN) & (ratings['Timestamp'] <= TS_MAX)]
log(f'ratings: dropped {before - len(ratings)} rows with invalid values')

# Convert epoch seconds -> proper datetime, then rename column to Date
ratings['Timestamp'] = pd.to_datetime(ratings['Timestamp'], unit='s')
ratings = ratings.rename(columns={'Timestamp': 'Date'})
log(f'ratings: converted Timestamp (epoch) -> Date (datetime); '
    f'range {ratings["Date"].min()} to {ratings["Date"].max()}')

ratings: dropped 0 rows with invalid values
ratings: converted Timestamp (epoch) -> Date (datetime); range 1995-01-09 11:46:49 to 2009-01-05 05:02:16


In [11]:
# Tags:
#   - UserID in 1..71567, MovieID in 1..65133
#   - Tag text is free-form; transliterate to ASCII and drop anything that
#     becomes empty after stripping
#   - Timestamp: same sanity bounds as ratings
before = len(tags)
for col in ['UserID', 'MovieID', 'Timestamp']:
    tags[col] = pd.to_numeric(tags[col], errors='coerce')
tags = tags.dropna(subset=['UserID', 'MovieID', 'Timestamp'])
tags = tags.astype({'UserID': int, 'MovieID': int, 'Timestamp': int})

tags = tags[(tags['UserID'] >= 1) & (tags['UserID'] <= 71567)]
tags = tags[(tags['MovieID'] >= 1) & (tags['MovieID'] <= 65133)]
tags = tags[(tags['Timestamp'] >= TS_MIN) & (tags['Timestamp'] <= TS_MAX)]

tag_changed = (~tags['Tag'].map(str.isascii)).sum()
tags['Tag'] = tags['Tag'].apply(to_ascii).str.strip()
tags = tags[tags['Tag'] != '']
log(f'tags: transliterated {tag_changed} tags to ASCII')
log(f'tags: dropped {before - len(tags)} rows with invalid values or empty tag')

tags['Timestamp'] = pd.to_datetime(tags['Timestamp'], unit='s')
tags = tags.rename(columns={'Timestamp': 'Date'})
log(f'tags: converted Timestamp (epoch) -> Date (datetime); '
    f'range {tags["Date"].min()} to {tags["Date"].max()}')

tags: transliterated 97 tags to ASCII
tags: dropped 0 rows with invalid values or empty tag
tags: converted Timestamp (epoch) -> Date (datetime); range 2005-12-23 04:49:47 to 2009-01-05 04:13:10


## 5. Referential integrity

Drop ratings and tags that reference a `MovieID` that doesn't exist in the movies table.
Drop movies that no rating references.

Note: per the README, tag UserIDs are drawn from a separate sample than rating UserIDs, so a tag UserID is not required to appear in `ratings`.

In [12]:
# Forward: drop ratings/tags pointing at a movie that doesn't exist in movies-1M table
before = len(ratings)
ratings = ratings[ratings['MovieID'].isin(movies['MovieID'])]
log(f'ratings: dropped {before - len(ratings)} rows with orphan MovieID')

before = len(tags)
tags = tags[tags['MovieID'].isin(movies['MovieID'])]
log(f'tags: dropped {before - len(tags)} rows with orphan MovieID')

# Reverse: drop movies-1M that no rating references
before = len(movies)
movies = movies[movies['MovieID'].isin(ratings['MovieID'])]
log(f'movies-1M: dropped {before - len(movies)} movies-1M with no ratings')

# Second pass for tags: pruning movies-1M may have orphaned some tags that pointed
# at movies-1M which existed in movies-1M.dat but had no ratings.
before = len(tags)
tags = tags[tags['MovieID'].isin(movies['MovieID'])]
log(f'tags: dropped {before - len(tags)} additional rows orphaned by movie pruning')

ratings: dropped 7 rows with orphan MovieID
tags: dropped 6 rows with orphan MovieID
movies: dropped 4 movies with no ratings
tags: dropped 30 additional rows orphaned by movie pruning


## 6. Validation tests

In [13]:
# Test 1: all ratings are multiples of 0.5 in [0.5, 5.0]
bad_ratings = ratings.loc[~ratings['Rating'].between(0.5, 5.0), 'Rating']
assert bad_ratings.empty, f"found ratings outside 0.5..5.0: {sorted(bad_ratings.unique())}"
non_half = ratings.loc[(ratings['Rating'] * 2) % 1 != 0, 'Rating']
assert non_half.empty, f"found non-half-star ratings: {sorted(non_half.unique())}"
log(f"PASS: all {len(ratings):,} ratings are half-star in [0.5, 5.0]")

# Test 2: every MovieID in ratings exists in the movies-1M table
orphan_movies = set(ratings['MovieID']) - set(movies['MovieID'])
assert not orphan_movies, f"{len(orphan_movies)} MovieIDs in ratings not in movies-1M table: {list(orphan_movies)[:5]}..."
log(f"PASS: all MovieIDs in ratings exist in movies-1M ({ratings['MovieID'].nunique():,} unique)")

# Test 3: every MovieID in tags exists in the movies-1M table
orphan_tag_movies = set(tags['MovieID']) - set(movies['MovieID'])
assert not orphan_tag_movies, f"{len(orphan_tag_movies)} MovieIDs in tags not in movies-1M table: {list(orphan_tag_movies)[:5]}..."
log(f"PASS: all MovieIDs in tags exist in movies-1M ({tags['MovieID'].nunique():,} unique)")

# Test 4: every non-empty genre token (across Genre1..GenreN cols) is in the whitelist
genre_cols = [c for c in movies.columns if c.startswith('Genre')]
all_genre_tokens = set(movies[genre_cols].values.ravel()) - {''}
unknown = all_genre_tokens - VALID_GENRES
assert not unknown, f"unknown genre tokens survived: {unknown}"
log(f"PASS: all movie genres are in the whitelist ({len(all_genre_tokens)} distinct)")

# Test 5: Year column is a plausible 4-digit release year (1800..next year)
import datetime as _dt
_next_year = _dt.date.today().year + 1
bad_years = movies.loc[(movies['Year'] < 1800) | (movies['Year'] > _next_year), 'Year']
assert bad_years.empty, f"{len(bad_years)} implausible years: {sorted(bad_years.unique())[:5]}"
log(f"PASS: all {len(movies):,} movies-1M have a valid Year ({movies['Year'].min()}..{movies['Year'].max()})")

# Test 6: ratings-per-user distribution. The README guarantees >=20 ratings per
# user in the RAW data; after cleaning some users may fall below that because
# their ratings pointed at movies-1M we dropped (no-genres, unknown genres, etc.),
# so this is an informational log rather than an assertion.
per_user = ratings.groupby('UserID').size()
low = per_user[per_user < 20]
log(f"INFO: ratings/user — min={per_user.min()}, median={int(per_user.median())}, "
    f"max={per_user.max()}; {len(low):,} users now below the raw-data floor of 20")

# Test 7: Titles, genre columns, and tags contain only ASCII characters
bad_title_chars = movies.loc[~movies['Title'].map(str.isascii), 'Title']
assert bad_title_chars.empty, f"{len(bad_title_chars)} titles with non-ASCII chars: {bad_title_chars.head().tolist()}"
for c in genre_cols:
    bad = movies.loc[~movies[c].map(str.isascii), c]
    assert bad.empty, f"{len(bad)} values in {c} with non-ASCII chars: {bad.head().tolist()}"
bad_tag_chars = tags.loc[~tags['Tag'].map(str.isascii), 'Tag']
assert bad_tag_chars.empty, f"{len(bad_tag_chars)} tags with non-ASCII chars: {bad_tag_chars.head().tolist()}"
log(f"PASS: all titles, genre columns, and tags use only English/numeric characters")

# Test 8: every (UserID, MovieID) tuple has exactly one rating
dup_pairs = ratings[ratings.duplicated(subset=['UserID', 'MovieID'], keep=False)]
assert dup_pairs.empty, f"{len(dup_pairs)} ratings share a (UserID, MovieID) with another"
log(f"PASS: every (UserID, MovieID) pair has exactly one rating ({len(ratings):,} unique pairs)")

PASS: all 10,000,047 ratings are half-star in [0.5, 5.0]


PASS: all MovieIDs in ratings exist in movies (10,676 unique)
PASS: all MovieIDs in tags exist in movies (7,596 unique)
PASS: all movie genres are in the whitelist (19 distinct)
PASS: all 10,676 movies have a valid Year (1915..2008)
INFO: ratings/user — min=20, median=69, max=7359; 0 users now below the raw-data floor of 20
PASS: all titles, genre columns, and tags use only English/numeric characters


PASS: every (UserID, MovieID) pair has exactly one rating (10,000,047 unique pairs)


## 7. Final summary & save

In [14]:
log('\n=== Final row counts ===')
log(f'movies-1M : {len(movies):>10,}')
log(f'ratings: {len(ratings):>10,}')
log(f'tags   : {len(tags):>10,}')
log(f'unique rating users: {ratings["UserID"].nunique():>6,}')
log(f'unique tag users   : {tags["UserID"].nunique():>6,}')


=== Final row counts ===
movies :     10,676
ratings: 10,000,047
tags   :     95,522
unique rating users: 69,878
unique tag users   :  4,009


In [15]:
movies.to_csv(OUT_DIR / 'movies_clean.csv', index=False)
ratings.to_csv(OUT_DIR / 'movies_ratings_clean.csv', index=False)
tags.to_csv(OUT_DIR / 'movies_tags_clean.csv', index=False)

with open(OUT_DIR / 'movies_cleaning_log.txt', 'w') as f:
    f.write('\n'.join(log_lines))

print(f'Saved cleaned CSVs and log to {OUT_DIR}/')

Saved cleaned CSVs and log to cleaned/
